In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
import joblib
import json

from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import make_scorer, roc_auc_score
from sklearn.preprocessing import StandardScaler

In [2]:
# Load preprocessed data
X = pd.read_csv('/content/drive/MyDrive/datasets/Heart_Disease_Project/preprocessed_features.csv')
y = pd.read_csv('/content/drive/MyDrive/datasets/Heart_Disease_Project/target_binary.csv').squeeze()

In [3]:
# Selected features
selected_features = joblib.load('/content/drive/MyDrive/datasets/Heart_Disease_Project/selected_features.pkl')
X = X[selected_features]
print(f"X shape: {X.shape}, y shape: {y.shape}")

X shape: (303, 11), y shape: (303,)


In [4]:
# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")


Train: (242, 11), Test: (61, 11)


In [5]:
# AUC scorer
auc_scorer = make_scorer(roc_auc_score, needs_proba=True)

In [6]:
# Base model (probability=True for AUC/proba)
svm_base = SVC(probability=True, random_state=42)
# Parameter grid (RBF focus; add 'linear' for comparison)
svm_param_grid = {
    'C': [0.1, 1, 10, 100],                 # Regularization
    'gamma': ['scale', 'auto', 0.01, 0.1],  # Kernel coeff
    'kernel': ['rbf', 'linear']             # Kernel type
}
# GridSearchCV
svm_grid = GridSearchCV(
    svm_base, svm_param_grid,
    cv=5, scoring=auc_scorer,
    n_jobs=-1, verbose=1
)
svm_grid.fit(X_train, y_train)  # Already scaled
# Results
best_svm = svm_grid.best_estimator_
best_params_svm = svm_grid.best_params_
cv_auc_svm = svm_grid.best_score_
print(f"SVM Best Params: {best_params_svm}")
print(f"SVM CV AUC: {cv_auc_svm:.3f}")
# Test
y_proba_svm = best_svm.predict_proba(X_test)[:, 1]
test_auc_svm = roc_auc_score(y_test, y_proba_svm)
print(f"SVM Test AUC: {test_auc_svm:.3f}")

Fitting 5 folds for each of 32 candidates, totalling 160 fits
SVM Best Params: {'C': 0.1, 'gamma': 'scale', 'kernel': 'rbf'}
SVM CV AUC: nan
SVM Test AUC: 0.958


In [7]:
# Save the best SVM model
model_filename ='/content/drive/MyDrive/datasets/Heart_Disease_Project/final_model_tuned.pkl'
joblib.dump(best_svm, model_filename)
print(f"Tuned SVM model saved: {model_filename}")

Tuned SVM model saved: /content/drive/MyDrive/datasets/Heart_Disease_Project/final_model_tuned.pkl
